In [1]:
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# !unzip '/content/drive/My Drive/dataset.zip' -d '/content/dataset'


In [1]:
!pip install tf-models-official

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import cv2

In [3]:
from official.projects.movinet.modeling import movinet
from official.projects.movinet.modeling import movinet_model
from official.projects.movinet.tools import export_saved_model

In [ ]:
print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [ ]:
# Check and display the number of GPUs available for model training

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


Configurations

In [ ]:
# --- Configurations ---
IMG_SIZE = 224
NUM_FRAMES = 30
BATCH_SIZE = 1
EPOCHS = 10
DATA_PATH = "/content/drive/My Drive/dataset"

Data Preparation

In [ ]:
# --- Data Preparation ---
def load_video_paths_and_labels(dataset_path):
    class_names = sorted(os.listdir(dataset_path))
    label_map = {name: idx for idx, name in enumerate(class_names)}

    video_paths = []
    labels = []

    for class_name in class_names:
        class_folder = os.path.join(dataset_path, class_name)
        for file in os.listdir(class_folder):
            if file.endswith(".mp4"):
                video_paths.append(os.path.join(class_folder, file))
                labels.append(label_map[class_name])

    return video_paths, labels, label_map



def sample_video_frames(path, num_frames=NUM_FRAMES, size=(IMG_SIZE, IMG_SIZE)):
    cap = cv2.VideoCapture(path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < num_frames:
        indices = np.linspace(0, total_frames-1, total_frames, dtype=int)
        pad = num_frames - total_frames
    else:
        indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
        pad = 0

    frames = []
    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        if i in indices:
            frame = cv2.resize(frame, size)
            frame = frame / 255.0
            frames.append(frame)

    cap.release()

    # Pad short videos with black frames
    while len(frames) < num_frames:
        frames.append(np.zeros((size[0], size[1], 3)))

    return np.array(frames)

def build_video_dataset(video_paths, labels):
    def generator():
        for path, label in zip(video_paths, labels):
            video = sample_video_frames(path)  
            yield video, label

    return tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )



Load and split dataset

In [ ]:
# Load and split dataset

video_paths, labels, label_map = load_video_paths_and_labels(DATA_PATH)
dataset = build_video_dataset(video_paths, labels)


X_val, X_temp, y_val, y_temp = train_test_split(video_paths, labels, test_size=0.9, stratify=labels, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_temp, y_temp, test_size=0.1, stratify=y_temp, random_state=42)

Build datasets

In [ ]:
# Build datasets
train_ds = build_video_dataset(X_train, y_train).shuffle(100).batch(2).prefetch(tf.data.AUTOTUNE)
val_ds   = build_video_dataset(X_val, y_val).batch(2).prefetch(tf.data.AUTOTUNE)
test_ds  = build_video_dataset(X_test, y_test).batch(2)

MoViNet Model Configuration

In [ ]:
# --- MoViNet Model Configuration ---
model_id = 'a2'
use_positional_encoding = model_id in {'a3', 'a4', 'a5'}
resolution = 224

backbone = movinet.Movinet(
    model_id=model_id,
    causal=True,
    conv_type='2plus1d',
    se_type='2plus3d',
    activation='hard_swish',
    gating_activation='hard_sigmoid',
    use_positional_encoding=use_positional_encoding,
    use_external_states=False,
)



model = movinet_model.MovinetClassifier(
    backbone,
    num_classes=600,
    output_states=True)


inputs = tf.ones([1, 13, 224, 224, 3])

# [Optional] Build the model and load a pretrained checkpoint.
model.build(inputs.shape)

In [ ]:
# Extract pretrained weights
!wget https://storage.googleapis.com/tf_model_garden/vision/movinet/movinet_a2_stream.tar.gz -O movinet_a2_stream.tar.gz -q
!tar -xvf movinet_a2_stream.tar.gz

checkpoint_dir = 'movinet_a2_stream'
checkpoint_path = tf.train.latest_checkpoint(checkpoint_dir)
checkpoint = tf.train.Checkpoint(model=model)
status = checkpoint.restore(checkpoint_path)
status.assert_existing_objects_matched()

movinet_a2_stream/
movinet_a2_stream/ckpt-1.data-00000-of-00001
movinet_a2_stream/ckpt-1.index
movinet_a2_stream/checkpoint


Finetuning for Crane Signals

In [ ]:
# --- Finetuning for Crane Signals ---
def build_classifier(batch_size, num_frames, resolution, backbone, num_classes, freeze_backbone=False):
  """Builds a classifier on top of a backbone model."""
  model = movinet_model.MovinetClassifier(
      backbone=backbone,
      num_classes=num_classes)
  if freeze_backbone:
    for layer in model.layers[-1:]:
      layer.trainable = True
    for layer in model.layers[:-1]:
      layer.trainable = False

  model.build([batch_size, num_frames, resolution, resolution, 3])

  return model

model = build_classifier(BATCH_SIZE, NUM_FRAMES, IMG_SIZE, backbone, 20, True)
loss_obj = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001)
# Load weights from the checkpoint to the rebuilt model
# model.load_weights('2_epoch.weights.h5')
model.compile(loss=loss_obj, optimizer='adam', metrics=['accuracy'])
# model.compile(loss=loss_obj, metrics=['accuracy'])

In [ ]:
checkpoint_path = "trained_model/.weights.h5"
checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                 save_weights_only=True,
                                                 verbose=1)


Training

In [ ]:
results = model.fit(train_ds,
                    epochs=10,
                    validation_data=val_ds,
                    validation_freq=1,
                    verbose=1,
#                     callbacks=[cp_callback]
)

Epoch 1/10
891/891 [==============================] - 1680s 2s/step - loss: 1.5295 - accuracy: 0.4905 - val_loss: 0.9372 - val_accuracy: 0.6727
Epoch 2/10
891/891 [==============================] - 936s 1000ms/step - loss: 0.6282 - accuracy: 0.7811 - val_loss: 0.7331 - val_accuracy: 0.7545
Epoch 3/10
891/891 [==============================] - 895s 954ms/step - loss: 0.3910 - accuracy: 0.8698 - val_loss: 0.6453 - val_accuracy: 0.7773
Epoch 4/10
891/891 [==============================] - 886s 947ms/step - loss: 0.2656 - accuracy: 0.9085 - val_loss: 0.5754 - val_accuracy: 0.8136
Epoch 5/10
891/891 [==============================] - 882s 940ms/step - loss: 0.2044 - accuracy: 0.9327 - val_loss: 0.7481 - val_accuracy: 0.8045
Epoch 6/10
891/891 [==============================] - 884s 945ms/step - loss: 0.1790 - accuracy: 0.9383 - val_loss: 0.5132 - val_accuracy: 0.8545
Epoch 7/10
891/891 [==============================] - 887s 948ms/step - loss: 0.1530 - accuracy: 0.9512 - val_loss: 0.5617 - 

Evaluate

In [ ]:
scores = model.evaluate(test_ds)

99/99 [==============================] - 156s 2s/step - loss: 0.5046 - accuracy: 0.8485


Save model

In [ ]:
# Save model
model.save("crane_model_checkpoint")
model.save("crane_model_checkpoint.keras")

files.download("crane_model_checkpoint")
files.download("crane_model_checkpoint.keras")





Metrics Visualization 


In [ ]:
# Store the training and validation accuracy and loss for visualization.

acc = results.history['accuracy']
val_acc = results.history['val_accuracy']

loss = results.history['loss']
val_loss = results.history['val_loss']

In [ ]:
# Visualize the training and validation accuracy and loss over the epochs.
plt.figure(figsize=(12, 6))

# Accuracy subplot
plt.subplot(1, 2, 1)
plt.plot(range(EPOCHS), acc, label="Training Accuracy")
plt.plot(range(EPOCHS), val_acc, label="Validation Accuracy")
plt.legend(loc='lower right')
plt.title("Training and Validation Accuracy")

# Loss subplot
plt.subplot(1, 2, 2)
plt.plot(range(EPOCHS), loss, label="Training Loss")
plt.plot(range(EPOCHS), val_loss, label="Validation Loss")
plt.legend(loc='upper right')
plt.title("Training and Validation Loss")

# Display the plots
plt.show()

Evaluate Precision, Recall, F1

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images)
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = labels.numpy()

    y_pred.extend(predicted_classes)
    y_true.extend(true_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')




1/1 [==============================] - 0s 204ms/step


In [ ]:
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.8592236467236468
Recall: 0.8484848484848485
F1 Score: 0.849390842851181


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get a batch from the test dataset
for batch in test_ds.take(4):
    inputs, labels = batch
    break

# Run prediction on the batch
predictions = model.predict(inputs)

# Convert logits to predicted class indices
predicted_classes = np.argmax(predictions, axis=1)

# Print results
for i in range(len(predicted_classes)):
    print(f"Video {i + 1}: Predicted = {predicted_classes[i]}, Actual = {labels[i].numpy()}")


1/1 [==============================] - 0s 268ms/step
Video 1: Predicted = 9, Actual = 7
Video 2: Predicted = 16, Actual = 16


Configuration To save The Model With Tensorflow Lite For Stream Mode

In [ ]:
# Rebuild the model from scratch
model_id = 'a2'
use_positional_encoding = model_id in {'a3', 'a4', 'a5'}
resolution = 224

backbone = movinet.Movinet(
    model_id=model_id,
    causal=True,
    conv_type='2plus1d',
    se_type='2plus3d',
    activation='hard_swish',
    gating_activation='hard_sigmoid',
    use_positional_encoding=use_positional_encoding,
    use_external_states=True,
)
# Note: this is a temporary model constructed for the
# purpose of loading the pre-trained checkpoint. Only
# the backbone will be used to build the custom classifier.

model = movinet_model.MovinetClassifier(
    backbone,
    num_classes=20,
    output_states=True)

inputs = tf.ones([1, 30, 224, 224, 3])
model.build(inputs.shape)

model.load_weights("/content/drive/My Drive/crane_model_checkpoint.keras")

In [ ]:
input_specs = {"image": tf.TensorSpec(shape=(1, 30, 224, 224, 3), dtype=tf.float32)}
state_specs = model.backbone.initial_state_specs(input_shape=input_specs["image"].shape)

# Create zero-initialized state inputs
state_inputs = {
    name: tf.zeros(spec.shape, dtype=spec.dtype)
    for name, spec in state_specs.items()
}

# Prepare final model input as a dict
final_input = {"image": inputs, **state_inputs}

# Run forward pass
_ = model(final_input)


In [ ]:

model.save("/content/drive/My Drive/crane_model_stream")

In [ ]:
saved_model_dir = '/content/drive/My Drive/crane_model_checkpoint'
tflite_filename = 'model_3.tflite'
input_shape = [1, 1, 224, 224, 3]


export_saved_model.export_saved_model(
    model=model,
    input_shape=input_shape,
    export_path=saved_model_dir,
    causal=True,
    bundle_input_init_states_fn=False)

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
tflite_model = converter.convert()

with open(tflite_filename, 'wb') as f:
  f.write(tflite_model)

Loading The Tensorflow Lite Model For Testing

In [ ]:


# Create the interpreter and signature runner
interpreter = tf.lite.Interpreter(model_path=tflite_filename)
runner = interpreter.get_signature_runner()

init_states = {
    name: tf.zeros(x['shape'], dtype=x['dtype'])
    for name, x in runner.get_input_details().items()
}
del init_states['image']

/usr/local/lib/python3.11/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
class_names = [None] * len(label_map)
for class_name, index in label_map.items():
    class_names[index] = class_name
print(class_names)



['Dog everything', 'Emergency Stop', 'Extend boom', 'Hoist', 'Lower', 'Lower boom', 'Lower boom and hoist loads', 'Move slowly', 'Raise boom', 'Raise boom and lower loads', 'Retract boom', 'Travel', 'Travel Both tracks', 'Travel only left track', 'Travel only right track', 'Use Auxiliary Hoist', 'Use Main Hoist', 'stop', 'swing boom left', 'swing boom right']


In [ ]:
def get_top_k(probs, k=5, label_map=class_names):
  #   """Outputs the top k model labels and probabilities on the given video."""
  top_predictions = tf.argsort(probs, axis=-1, direction='DESCENDING')[:k]
  top_labels = tf.gather(label_map, top_predictions, axis=-1)
  top_labels = [label.decode('utf8') for label in top_labels.numpy()]
  top_probs = tf.gather(probs, top_predictions, axis=-1).numpy()
  return tuple(zip(top_labels, top_probs))

In [ ]:
import imageio

def to_gif(frames, filename='output.gif', fps=5):
    """
    Convert a sequence of frames to a GIF.

    Args:
        frames (numpy.ndarray): Array of shape (T, H, W, 3)
        filename (str): Output GIF filename
        fps (int): Frames per second
    """
    frames_uint8 = (frames * 255).astype(np.uint8) if frames.max() <= 1.0 else frames.astype(np.uint8)
    imageio.mimsave(filename, frames_uint8, fps=fps)
    print(f"GIF saved to {filename}")


In [ ]:
import random

# Get 10 random examples from test_ds
test_videos = list(test_ds.take(100))  # Take more to enable randomness
random.shuffle(test_videos)
test_videos = test_videos[:10]

for i, (frames, label) in enumerate(test_videos):
    print(f"\n📹 Testing Video {i+1}...")

    states = init_states
    for clip in frames[0]:  # iterate through each [1, 1, H, W, 3] frame
        outputs = runner(**states, image=clip)
        logits = outputs.pop('logits')[0]
        states = outputs

    # Predict
    probs = tf.nn.softmax(logits)
    top_k = get_top_k(probs)

    # Show top predictions
    for label_name, prob in top_k:
        print(f"{label_name}: {prob:.5f}")

    # Show ground truth
    true_label_index = int(label.numpy()[0]) if len(label.shape) > 0 else int(label.numpy())
    print(f"🎯 Ground Truth: {class_names[true_label_index]}")

    # Optional: Save GIF
    gif_filename = f"output_{i+1}.gif"
    to_gif(frames[0].numpy(), filename=gif_filename)
    print(f"📁 GIF saved to {gif_filename}")



📹 Testing Video 1...
Retract boom: 0.99998
Extend boom: 0.00002
Use Auxiliary Hoist: 0.00000
Travel only left track: 0.00000
Dog everything: 0.00000
🎯 Ground Truth: Retract boom
GIF saved to output_1.gif
📁 GIF saved to output_1.gif

📹 Testing Video 2...
Lower: 0.78009
Travel only right track: 0.15456
Retract boom: 0.02565
Extend boom: 0.01881
stop: 0.01353
🎯 Ground Truth: Lower
GIF saved to output_2.gif
📁 GIF saved to output_2.gif

📹 Testing Video 3...
Travel only left track: 0.99561
Extend boom: 0.00213
Hoist: 0.00102
Travel: 0.00045
Retract boom: 0.00038
🎯 Ground Truth: Travel only left track
GIF saved to output_3.gif
📁 GIF saved to output_3.gif

📹 Testing Video 4...
swing boom right: 0.99119
Hoist: 0.00591
swing boom left: 0.00113
Travel: 0.00069
Raise boom: 0.00053
🎯 Ground Truth: swing boom right
GIF saved to output_4.gif
📁 GIF saved to output_4.gif

📹 Testing Video 5...
stop: 1.00000
Lower boom: 0.00000
Dog everything: 0.00000
Emergency Stop: 0.00000
swing boom left: 0.00000
🎯 G

Configuration To save The Model With Tensorflow Lite For Stream Mode

In [5]:
model_id = 'a2'
use_positional_encoding = model_id in {'a3', 'a4', 'a5'}
resolution = 224

backbone = movinet.Movinet(
    model_id=model_id,
    causal=False,
    conv_type='2plus1d',
    se_type='2plus3d',
    activation='hard_swish',
    gating_activation='hard_sigmoid',
    use_positional_encoding=use_positional_encoding,
    use_external_states=False,
)
# Note: this is a temporary model constructed for the
# purpose of loading the pre-trained checkpoint. Only
# the backbone will be used to build the custom classifier.

model = movinet_model.MovinetClassifier(
    backbone,
    num_classes=20,
    output_states=False)

inputs = tf.ones([1, 30, 224, 224, 3])

model.build(inputs.shape)
model.load_weights("/content/drive/My Drive/crane_model_checkpoint.keras")

In [7]:
# Run a dummy forward pass
inputs = tf.ones([1, 30, 224, 224, 3])
_ = model(inputs)
model_save_path = "/content/drive/My Drive/crane_model_base30"
model.save(model_save_path)


In [8]:
tflite_filename = "/content/drive/My Drive/model_3_b30.tflite"

converter = tf.lite.TFLiteConverter.from_saved_model(model_save_path)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(tflite_filename, "wb") as f:
    f.write(tflite_model)
